In [ ]:
import sys
import os
import subprocess
from pathlib import Path

# Ensure project import path
PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
# Set working directory so relative paths (e.g., src/config/*.yaml) resolve
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline

# CuPy is required - ensure CUDA_PATH and LD_LIBRARY_PATH are set
# This allows the notebook to work even if Jupyter wasn't started with modules loaded
if "CUDA_PATH" not in os.environ:
    print("CUDA_PATH not set, attempting to load modules...")
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True,
            executable='/bin/bash',
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"✓ Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
        else:
            print(f"⚠ Could not load modules. Error: {result.stderr}")
            raise RuntimeError(
                "CUDA_PATH not set and could not load modules. "
                "Please run: module load cuda/12.2 before starting Jupyter."
            )
    except Exception as e:
        print(f"✗ Could not load modules: {e}")
        raise RuntimeError(
            f"Failed to load CUDA modules: {e}\n"
            "Please ensure CUDA is loaded before starting Jupyter:\n"
            "  module load cuda/12.2"
        ) from e

# Ensure LD_LIBRARY_PATH includes CUDA library directory for NVRTC (libnvrtc.so.12)
# This is required for CuPy to compile kernels at runtime
# Note: Setting this BEFORE importing CuPy is critical
cuda_path = os.environ.get('CUDA_PATH')
libnvrtc_path = None

if cuda_path:
    # Check both standard location and Compute Canada's targets/x86_64-linux/lib location
    cuda_lib_paths = [
        os.path.join(cuda_path, 'lib64'),  # Standard location
        os.path.join(cuda_path, 'targets', 'x86_64-linux', 'lib'),  # Compute Canada location
    ]
    
    current_ld_path = os.environ.get('LD_LIBRARY_PATH', '')
    ld_paths = current_ld_path.split(':') if current_ld_path else []
    paths_added = []
    
    # Find which paths exist and add them to LD_LIBRARY_PATH
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            if cuda_lib_path not in ld_paths:
                paths_added.append(cuda_lib_path)
                ld_paths.insert(0, cuda_lib_path)  # Prepend for priority
    
    if paths_added:
        os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
        print(f"✓ Updated LD_LIBRARY_PATH to include: {', '.join(paths_added)}")
    else:
        # Check if paths were already included
        found_paths = [p for p in cuda_lib_paths if p in ld_paths]
        if found_paths:
            print(f"✓ LD_LIBRARY_PATH already includes CUDA libraries: {', '.join(found_paths)}")
    
    # Find libnvrtc.so.12 and preload it using ctypes
    # This ensures CuPy can find it even if LD_LIBRARY_PATH isn't fully respected
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            potential_libnvrtc = os.path.join(cuda_lib_path, 'libnvrtc.so.12')
            if os.path.exists(potential_libnvrtc):
                libnvrtc_path = potential_libnvrtc
                print(f"✓ Found libnvrtc.so.12 at: {libnvrtc_path}")
                
                # Preload the library using ctypes so CuPy can find it
                # Use RTLD_GLOBAL to make symbols available to other libraries
                try:
                    import ctypes
                    # Try multiple loading strategies
                    try:
                        # Strategy 1: Load with full path and RTLD_GLOBAL
                        lib = ctypes.CDLL(libnvrtc_path, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)")
                    except Exception as e1:
                        # Strategy 2: Try without RTLD_GLOBAL
                        try:
                            lib = ctypes.CDLL(libnvrtc_path)
                            print(f"✓ Preloaded libnvrtc.so.12 using ctypes (standard)")
                        except Exception as e2:
                            raise e1 from e2
                except Exception as e:
                    print(f"⚠ Warning: Could not preload libnvrtc.so.12: {e}")
                    print(f"  CuPy may still work if LD_LIBRARY_PATH is set correctly")
                    print(f"  You may need to restart the Jupyter kernel with:")
                    print(f"    export LD_LIBRARY_PATH={os.path.dirname(libnvrtc_path)}:$LD_LIBRARY_PATH")
                
                # Verify that ctypes can find the library by name (as CuPy will try)
                try:
                    import ctypes.util
                    found_lib = ctypes.util.find_library('nvrtc')
                    if found_lib:
                        print(f"✓ ctypes.util.find_library('nvrtc') found: {found_lib}")
                    else:
                        print(f"⚠ ctypes.util.find_library('nvrtc') returned None")
                        print(f"  This may cause issues. Try loading by name:")
                        try:
                            test_lib = ctypes.CDLL('libnvrtc.so.12')
                            print(f"✓ Successfully loaded libnvrtc.so.12 by name")
                        except Exception as name_err:
                            print(f"✗ Failed to load libnvrtc.so.12 by name: {name_err}")
                            print(f"  You MUST restart the Jupyter kernel with LD_LIBRARY_PATH set")
                except Exception as diag_err:
                    print(f"⚠ Could not run diagnostics: {diag_err}")
                break
    
    if not libnvrtc_path:
        # Try to find any version of libnvrtc.so
        import glob
        for cuda_lib_path in cuda_lib_paths:
            if os.path.exists(cuda_lib_path):
                nvrtc_files = glob.glob(os.path.join(cuda_lib_path, 'libnvrtc.so*'))
                if nvrtc_files:
                    # Try to use the most specific version
                    nvrtc_files.sort(reverse=True)  # Prefer .so.12.2.140 over .so.12 over .so
                    potential_lib = nvrtc_files[0]
                    print(f"⚠ libnvrtc.so.12 not found, but found: {nvrtc_files}")
                    print(f"  Attempting to use: {potential_lib}")
                    try:
                        import ctypes
                        ctypes.CDLL(potential_lib, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded {potential_lib} using ctypes")
                        libnvrtc_path = potential_lib
                    except Exception as e:
                        print(f"⚠ Could not preload {potential_lib}: {e}")
                    break
        else:
            print(f"⚠ Warning: libnvrtc.so.12 not found in any CUDA library directory")
            print(f"  This may cause CuPy kernel compilation to fail")
else:
    print("⚠ CUDA_PATH not set, cannot configure LD_LIBRARY_PATH")


from src.utils.array_backend import np, random, is_cupy
from src.classes.belief_mdp_n import BeliefMDP_n_Localization
from src.classes.model import DoubleIntegratorModel, LIDAR
from src.classes.mapping import LidarGridMapVec
from src.utils.map import load_obstacles_config
from tqdm import tqdm
import time
import warnings

# Suppress CuPy experimental FutureWarnings for multivariate_normal
# These warnings are harmless and clutter the output
warnings.filterwarnings('ignore', category=FutureWarning, module='cupy.random')

print(f"✓ All imports successful")
print(f"Using backend: {'CuPy (GPU)' if is_cupy else 'NumPy (CPU)'}")

# Verify we're using CuPy
if not is_cupy:
    raise RuntimeError(
        "CuPy is required but not being used. "
        "Check CUDA installation and CuPy setup."
    )

"""
Verification tests for Monte Carlo integration (η_n) in BeliefMDP_n_Localization.

This test suite addresses:
1. MC convergence and suitable n_samples determination
2. Batch size optimization for GPU utilization
3. F, H, and η_n mathematical consistency

For Localization:
- Belief space: π(x) - shape (m_n,)
- Map is known (not part of belief state)
- Uses obstacle segments directly (no occupancy grid)

Uses the same model setup as SLAM notebook:
- DoubleIntegratorModel with n=2, dt=1.0, max_a=2.0
- LIDAR(fov=360, r_max=10.0, B=4)
"""


In [ ]:
def test_mc_convergence_eta_n(batch_size=1000, quantization_level=2, 
                               min_samples=1000, max_samples=10000000, initial_samples=1000,
                               rel_error_threshold=0.05, ci_width_threshold=0.01, 
                               variance_stability_threshold=0.1, z_score=1.96):
    """
    Test 1: Determine suitable n_samples for η_n via robust statistical convergence analysis (Localization).

    Uses multiple convergence criteria:
    1. Relative error (coefficient of variation): CV = std/mean < threshold
    2. Confidence interval width: 95% CI width < threshold
    3. Variance stabilization: variance change < threshold
    4. Binomial approximation: np > 5 and n(1-p) > 5

    Args:
        batch_size: Batch size to use for η_n computation (default: 1000)
        quantization_level: State quantization level (default: 2)
        min_samples: Minimum number of samples before checking convergence (default: 1000)
        max_samples: Maximum number of samples to try (default: 10000000)
        initial_samples: Initial number of samples (default: 1000)
        rel_error_threshold: Maximum relative error (CV) allowed (default: 0.05 = 5%)
        ci_width_threshold: Maximum 95% CI width allowed (default: 0.01 = 1%)
        variance_stability_threshold: Maximum relative change in variance for stability (default: 0.1 = 10%)
        z_score: Z-score for confidence interval (1.96 for 95% CI, 2.576 for 99% CI)
    
    Returns:
        dict: Dictionary with convergence statistics and results
    """
    obstacles, area = load_obstacles_config(environment='toy2')

    motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
    sensor = LIDAR(fov=360, r_max=10.0, B=4)
    grid_map = LidarGridMapVec(
        x_min=area[0], x_max=area[1],
        y_min=area[2], y_max=area[3],
        quantization_level=quantization_level,  
    )
    bmdp = BeliefMDP_n_Localization(
        n=quantization_level, 
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=obstacles,
        _map=grid_map,
    )
    bmdp.set_known_map(obstacles)

    # Create test beliefs and actions
    m_n = bmdp.SQ.m_n
    
    # Get actions from action space
    actions = bmdp.AQ.U  # All actions from action quantizer
    n_actions = bmdp.AQ.n_u
    print(f"\nAction space: {n_actions} actions available")
    
    # Create multiple initial beliefs for testing
    # Test beliefs concentrated at different states
    initial_beliefs = []
    belief_names = []
    
    # 1. Concentrated at middle state
    π_mid = np.zeros(m_n)
    π_mid[m_n // 2] = 1.0
    initial_beliefs.append(π_mid)
    belief_names.append("middle_state")
    
    # 2. Concentrated at corner state (if available)
    if m_n > 1:
        π_corner = np.zeros(m_n)
        π_corner[0] = 1.0
        initial_beliefs.append(π_corner)
        belief_names.append("corner_state")
    
    # 3. Uniform over states
    π_uniform = np.ones(m_n) / m_n
    initial_beliefs.append(π_uniform)
    belief_names.append("uniform")
    
    print(f"Testing {len(initial_beliefs)} initial beliefs × {n_actions} actions = {len(initial_beliefs) * n_actions} combinations")

    print(f"\n=== Testing η_n MC Convergence (Robust Statistical Analysis) - Localization ===")
    print(f"Quantization level: {quantization_level}")
    print(f"Belief size: {m_n} poses")
    print(f"Using batch_size: {batch_size}")
    print(f"\nConvergence criteria:")
    print(f"  1. Relative error (CV): < {rel_error_threshold*100:.1f}%")
    print(f"  2. 95% CI width: < {ci_width_threshold*100:.1f}%")
    print(f"  3. Variance stability: < {variance_stability_threshold*100:.1f}% change")
    print(f"  4. Binomial approximation: np > 5 and n(1-p) > 5")
    
    # Store results for all combinations
    all_results = []

    # Test each combination of initial belief and action
    for belief_idx, (π_0, belief_name) in enumerate(zip(initial_beliefs, belief_names)):
        for action_idx, u in enumerate(actions):
            print(f"\n{'='*70}")
            print(f"Testing: Belief '{belief_name}' ({belief_idx+1}/{len(initial_beliefs)}), "
                  f"Action {action_idx+1}/{n_actions} u={u}")
            print(f"{'='*70}")
            
            # Sample multiple observations from the belief-action pair to find a non-extreme target
            n_candidate_samples = 20
            y_samples = bmdp.sample_observations_batch(π_0, u, n_candidate_samples)
            
            # Filter each observation to get candidate target beliefs
            candidate_targets = bmdp.F_batch(π_0, u, y_samples)  # (n_candidate_samples, m_n)
            
            # Quick check: find a target that gives non-extreme probability
            π_target = None
            prob_preview = None
            extreme_tolerance = 1e-3
            
            # Try candidates to find one with non-extreme probability
            for candidate_idx in range(n_candidate_samples):
                candidate_target = candidate_targets[candidate_idx]
                # Quick preview with small sample size to check if probability is extreme
                prob_preview = bmdp.η_n(
                    candidate_target, π_0, u,
                    n_samples=2000,
                    seed=42,
                    batch_size=batch_size,
                    show_progress=False
                )
                
                # If probability is non-extreme, use this target
                if extreme_tolerance < prob_preview < (1.0 - extreme_tolerance):
                    π_target = candidate_target
                    print(f"Selected candidate {candidate_idx+1} with preview probability: {prob_preview:.6f}")
                    break
            
            # If no moderate probability found, skip convergence test for this combination
            if π_target is None:
                print(f"⚠ All {n_candidate_samples} candidates have extreme probabilities (preview: {prob_preview:.6f})")
                print(f"  Skipping convergence test - extreme probabilities are deterministic (already converged)")
                
                result = {
                    'belief_name': belief_name,
                    'belief_idx': belief_idx,
                    'action_idx': action_idx,
                    'action': u.get() if hasattr(u, 'get') else u,
                    'estimates': [prob_preview],
                    'n_samples_list': [2000],
                    'std_errors': [],
                    'ci_widths': [],
                    'rel_errors': [],
                    'final_estimate': prob_preview,
                    'converged': True,
                    'is_extreme': True,
                    'final_n_samples': 2000,
                    'skipped': True
                }
                
                print(f"\n--- Results for {belief_name}, action {action_idx+1} ---")
                print(f"  Final estimate: {prob_preview:.6e} (extreme probability)")
                print(f"  Status: Skipped convergence test - extreme probability")
                print(f"  Converged: ✓ (extreme probabilities are deterministic)")
                
                all_results.append(result)
                continue
            
            # Run convergence test for this combination
            estimates = []
            n_samples_list = []
            std_errors = []
            ci_widths = []
            rel_errors = []
            n_samples = initial_samples
            converged = False
            prev_variance = None

            print(f"Convergence progress:")
            while n_samples <= max_samples:
                # Compute estimate
                prob = bmdp.η_n(
                    π_target, π_0, u, 
                    n_samples=n_samples, 
                    seed=42, 
                    batch_size=batch_size,
                    show_progress=False
                )
                
                estimates.append(prob)
                n_samples_list.append(n_samples)
                
                # Compute statistical measures
                if prob > 0 and prob < 1:
                    # Standard error for binomial estimator
                    std_error = np.sqrt(prob * (1 - prob) / n_samples)
                    std_errors.append(std_error)
                    
                    # Confidence interval width (z_score * 2 * std_error)
                    ci_width = z_score * 2 * std_error
                    ci_widths.append(ci_width)
                    
                    # Relative error (coefficient of variation)
                    rel_error = std_error / prob if prob > 0 else float('inf')
                    rel_errors.append(rel_error)
                    
                    # Variance (for stability check)
                    variance = prob * (1 - prob) / n_samples
                else:
                    std_errors.append(0.0)
                    ci_widths.append(0.0)
                    rel_errors.append(0.0 if prob == 0 or prob == 1 else float('inf'))
                    variance = 0.0
                
                # Print progress with statistical measures
                print(f"  n_samples={n_samples:10d}: η_n = {prob:.6e}", end="")
                if prob > 0 and prob < 1:
                    print(f" | std={std_errors[-1]:.6e} | CV={rel_errors[-1]*100:.2f}% | CI_width={ci_widths[-1]:.6e}")
                else:
                    print()
                
                # Check convergence criteria (only after minimum samples)
                if n_samples >= min_samples:
                    convergence_checks = []
                    
                    # Criterion 1: Relative error (CV)
                    if prob > 0 and prob < 1:
                        cv_ok = rel_errors[-1] < rel_error_threshold
                        convergence_checks.append(('CV', cv_ok, f"{rel_errors[-1]*100:.2f}% < {rel_error_threshold*100:.1f}%"))
                    else:
                        cv_ok = True
                        convergence_checks.append(('CV', cv_ok, "extreme probability"))
                    
                    # Criterion 2: Confidence interval width
                    if prob > 0 and prob < 1:
                        ci_ok = ci_widths[-1] < ci_width_threshold
                        convergence_checks.append(('CI_width', ci_ok, f"{ci_widths[-1]:.6e} < {ci_width_threshold:.6e}"))
                    else:
                        ci_ok = True
                        convergence_checks.append(('CI_width', ci_ok, "extreme probability"))
                    
                    # Criterion 3: Variance stabilization
                    if prev_variance is not None and variance > 0:
                        variance_change = abs(variance - prev_variance) / prev_variance if prev_variance > 0 else float('inf')
                        var_stable = variance_change < variance_stability_threshold
                        convergence_checks.append(('Variance_stability', var_stable, 
                                                 f"{variance_change*100:.2f}% < {variance_stability_threshold*100:.1f}%"))
                    else:
                        var_stable = False
                        convergence_checks.append(('Variance_stability', var_stable, "need more samples"))
                    
                    # Criterion 4: Binomial approximation validity
                    binomial_ok = (n_samples * prob >= 5) and (n_samples * (1 - prob) >= 5)
                    convergence_checks.append(('Binomial_approx', binomial_ok, 
                                             f"np={n_samples*prob:.1f}, n(1-p)={n_samples*(1-prob):.1f}"))
                    
                    # Check if all criteria are met
                    all_criteria_met = all(check[1] for check in convergence_checks)
                    
                    if all_criteria_met:
                        converged = True
                        print(f"\n✓ Converged! All criteria met:")
                        for name, passed, info in convergence_checks:
                            status = "✓" if passed else "✗"
                            print(f"  {status} {name}: {info}")
                        break
                    elif n_samples >= min_samples * 2:
                        print(f"    Convergence status:")
                        for name, passed, info in convergence_checks:
                            status = "✓" if passed else "✗"
                            print(f"      {status} {name}: {info}")
                
                prev_variance = variance
                
                # Increase sample count conservatively based on current error
                if prob > 0 and prob < 1:
                    current_rel_error = rel_errors[-1] if rel_errors else float('inf')
                    if current_rel_error > rel_error_threshold * 2:
                        error_reduction_factor = 1.3
                    elif current_rel_error > rel_error_threshold:
                        error_reduction_factor = 1.2
                    else:
                        error_reduction_factor = 1.1
                    n_samples = int(n_samples * error_reduction_factor)
                else:
                    if n_samples < 100000:
                        n_samples = int(n_samples * 1.1)
                    elif n_samples < 1000000:
                        n_samples = int(n_samples * 1.15)
                    else:
                        n_samples = int(n_samples * 1.2)
                
                n_samples = min(n_samples, max_samples)

            # Store results for this combination
            final_estimate = estimates[-1] if estimates else 0.0
            extreme_tolerance = 1e-6
            is_extreme = (abs(final_estimate - 1.0) < extreme_tolerance or abs(final_estimate - 0.0) < extreme_tolerance)
            
            if not converged:
                if is_extreme:
                    print(f"\n⚠ Extreme probability ({final_estimate:.6e}) - convergence criteria not applicable")
                    print(f"   Extreme probabilities are already well-estimated (no variance)")
                    converged = True
                else:
                    print(f"\n⚠ Did not converge within {max_samples} samples")
                    if len(estimates) >= 1 and prob > 0 and prob < 1:
                        print(f"   Final relative error: {rel_errors[-1]*100:.2f}% (target: < {rel_error_threshold*100:.1f}%)")
                        print(f"   Final CI width: {ci_widths[-1]:.6e} (target: < {ci_width_threshold:.6e})")
            
            result = {
                'belief_name': belief_name,
                'belief_idx': belief_idx,
                'action_idx': action_idx,
                'action': u.get() if hasattr(u, 'get') else u,
                'estimates': estimates,
                'n_samples_list': n_samples_list,
                'std_errors': std_errors,
                'ci_widths': ci_widths,
                'rel_errors': rel_errors,
                'final_estimate': final_estimate,
                'converged': converged,
                'is_extreme': is_extreme,
                'final_n_samples': n_samples_list[-1] if n_samples_list else 0,
                'skipped': False
            }
            
            # Print summary for this combination
            if is_extreme:
                print(f"\n--- Results for {belief_name}, action {action_idx+1} ---")
                print(f"  Final estimate: {final_estimate:.6e} (extreme probability)")
                print(f"  Samples used: {result['final_n_samples']:,}")
                print(f"  Status: Extreme probability - convergence criteria not applicable")
                print(f"  Converged: ✓ (extreme probabilities are deterministic)")
            elif final_estimate > 0 and final_estimate < 1:
                final_std = std_errors[-1] if std_errors else 0.0
                final_rel_error = rel_errors[-1] if rel_errors else float('inf')
                final_ci_width = ci_widths[-1] if ci_widths else 0.0
                
                print(f"\n--- Results for {belief_name}, action {action_idx+1} ---")
                print(f"  Final estimate: {final_estimate:.6e}")
                print(f"  Samples used: {result['final_n_samples']:,}")
                print(f"  Relative error: {final_rel_error*100:.2f}%")
                print(f"  95% CI width: {final_ci_width:.6e}")
                print(f"  Converged: {'✓' if converged else '✗'}")
            
            all_results.append(result)
    
    # Aggregate results across all combinations
    print(f"\n{'='*70}")
    print(f"=== Aggregate Results Across All Combinations ===")
    print(f"{'='*70}")
    
    converged_count = sum(1 for r in all_results if r['converged'])
    extreme_count = sum(1 for r in all_results if r['is_extreme'])
    skipped_count = sum(1 for r in all_results if r.get('skipped', False))
    non_extreme_converged = sum(1 for r in all_results if r['converged'] and not r['is_extreme'])
    non_extreme_total = sum(1 for r in all_results if not r['is_extreme'])
    tested_count = sum(1 for r in all_results if not r.get('skipped', False))
    total_combinations = len(all_results)
    
    print(f"\nConvergence summary:")
    print(f"  Total combinations: {total_combinations}")
    print(f"  Skipped (extreme probabilities): {skipped_count} (converged by definition)")
    print(f"  Tested: {tested_count}")
    if tested_count > 0:
        print(f"  Non-extreme probabilities tested: {non_extreme_total}")
        print(f"  Non-extreme converged: {non_extreme_converged}/{non_extreme_total} "
              f"({non_extreme_converged/non_extreme_total*100:.1f}% of tested non-extreme)" if non_extreme_total > 0 else "")
    print(f"  Overall converged: {converged_count}/{total_combinations} ({converged_count/total_combinations*100:.1f}%)")
    
    # Statistics on required samples (only for non-extreme converged cases that were tested)
    final_samples_list = [r['final_n_samples'] for r in all_results 
                          if r['converged'] and not r['is_extreme'] and not r.get('skipped', False)]
    if final_samples_list:
        final_samples_arr = np.array(final_samples_list)
        print(f"\nRequired samples (for non-extreme converged cases):")
        print(f"  Mean: {float(np.mean(final_samples_arr)):,.0f}")
        print(f"  Median: {float(np.median(final_samples_arr)):,.0f}")
        print(f"  Min: {int(np.min(final_samples_arr)):,}")
        print(f"  Max: {int(np.max(final_samples_arr)):,}")
    
    # Show which combinations converged
    print(f"\nConvergence by combination:")
    for r in all_results:
        status = "✓" if r['converged'] else "✗"
        if r.get('skipped', False):
            status_marker = " [SKIPPED - EXTREME]"
        elif r['is_extreme']:
            status_marker = " [EXTREME]"
        else:
            status_marker = ""
        u_str = f"[{r['action'][0]:.3f}, {r['action'][1]:.3f}]"
        print(f"  {status} {r['belief_name']:15s} | action {r['action_idx']+1:2d} {u_str:20s} | "
              f"n_samples={r['final_n_samples']:8,} | η_n={r['final_estimate']:.6e}{status_marker}")
    
    # Return aggregate results
    mean_samples_converged = None
    if final_samples_list:
        final_samples_arr = np.array(final_samples_list)
        mean_samples_converged = float(np.mean(final_samples_arr))
    
    return {
        'all_results': all_results,
        'converged_count': converged_count,
        'total_combinations': total_combinations,
        'convergence_rate': converged_count / total_combinations if total_combinations > 0 else 0.0,
        'mean_samples_converged': mean_samples_converged,
        'batch_size': batch_size
    }

print("\n" + "="*70)
print("Running MC convergence test for Localization...")
print("="*70)
convergence_results = test_mc_convergence_eta_n(quantization_level=2)


In [ ]:
def get_gpu_memory_info():
    """
    Get GPU memory information using nvidia-smi and CuPy memory pool.
    
    Returns:
        dict with memory stats: total, used, free, cupy_used, cupy_free
    """
    import subprocess
    
    stats = {}
    
    # Get GPU memory from nvidia-smi
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=memory.total,memory.used,memory.free', 
             '--format=csv,nounits,noheader'],
            capture_output=True,
            text=True,
            timeout=5
        )
        if result.returncode == 0:
            lines = result.stdout.strip().split('\n')
            if lines:
                values = [int(x.strip()) for x in lines[0].split(',')]
                stats['total_mb'] = values[0]
                stats['used_mb'] = values[1]
                stats['free_mb'] = values[2]
    except Exception as e:
        print(f"⚠ Could not query nvidia-smi: {e}")
    
    # Get CuPy memory pool stats
    if is_cupy:
        try:
            mempool = np.get_default_memory_pool()
            stats['cupy_used_mb'] = mempool.used_bytes() / (1024**2)
            stats['cupy_free_mb'] = mempool.free_bytes() / (1024**2)
            stats['cupy_total_mb'] = stats['cupy_used_mb'] + stats['cupy_free_mb']
        except Exception as e:
            print(f"⚠ Could not query CuPy memory pool: {e}")
    
    return stats


def test_optimal_batch_size(quantization_level=2, test_batch_sizes=None, test_n_samples=None):
    """
    Test 2: Find optimal batch size for η_n based on GPU memory usage and performance (Localization).

    Tests different batch sizes to find the optimal one that maximizes GPU utilization
    without causing out-of-memory errors. Uses the adequate n_samples determined from Test 1.

    Args:
        quantization_level: State quantization level (affects memory usage)
        test_batch_sizes: List of batch sizes to test (default: [500, 1000, 2000, 5000, 10000, ...])
        test_n_samples: Number of samples to use (default: use mean_samples_converged from convergence test)
    
    Returns:
        int: Optimal batch size
    """
    # Use the convergence results to determine adequate n_samples
    if test_n_samples is None:
        if 'convergence_results' in globals() and convergence_results.get('mean_samples_converged'):
            test_n_samples = int(convergence_results['mean_samples_converged'])
            print(f"Using n_samples={test_n_samples:,} from convergence test results")
        else:
            test_n_samples = 1000000  # Default fallback
            print(f"⚠ No convergence results found, using default n_samples={test_n_samples:,}")
    
    # Check GPU state gracefully
    gpu_state_ok = True
    if is_cupy:
        try:
            np.cuda.Device(0).use()
            test_array = np.array([1.0, 2.0, 3.0])
            _ = test_array.sum()
            del test_array
        except Exception as e:
            gpu_state_ok = False
            print(f"⚠ GPU state check failed: {e}")
    
    obstacles, area = load_obstacles_config(environment='toy2')

    try:
        motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
        sensor = LIDAR(fov=360, r_max=10.0, B=4)
        grid_map = LidarGridMapVec(
            x_min=area[0], x_max=area[1],
            y_min=area[2], y_max=area[3],
            quantization_level=quantization_level,  
        )
        bmdp = BeliefMDP_n_Localization(
            n=quantization_level, 
            motion_model=motion_model,
            measurement_model=sensor,
            obstacles=obstacles,
            _map=grid_map,
        )
        bmdp.set_known_map(obstacles)
    except Exception as e:
        error_msg = str(e)
        if "CUDA" in error_msg or "cuda" in error_msg or "IllegalAddress" in error_msg:
            print(f"\n✗ GPU Error during model creation: {error_msg}")
            print("\n" + "="*70)
            print("GPU STATE CORRUPTION DETECTED")
            print("="*70)
            print("\nSOLUTION: Restart the Jupyter kernel (Kernel -> Restart)")
            raise RuntimeError("GPU is in a corrupted state. Please restart the Jupyter kernel.") from e
        else:
            raise

    # Create test beliefs
    m_n = bmdp.SQ.m_n

    # Concentrated prior
    π_0 = np.zeros(m_n)
    π_0[m_n // 2] = 1.0

    # Uniform target
    π_target = np.ones(m_n) / m_n

    u = np.array([0.0, 0.0])

    print(f"\n=== Testing Optimal Batch Size (Localization) ===")
    print(f"Quantization level: {quantization_level}")
    print(f"Belief size: {m_n} poses")
    print(f"Using n_samples: {test_n_samples:,}")

    # Get initial GPU memory state
    if is_cupy:
        print(f"\n=== GPU Memory Information ===")
        initial_mem = get_gpu_memory_info()
        if 'total_mb' in initial_mem:
            print(f"GPU Total Memory: {initial_mem['total_mb']:.0f} MB")
            print(f"GPU Used Memory: {initial_mem['used_mb']:.0f} MB")
            print(f"GPU Free Memory: {initial_mem['free_mb']:.0f} MB")
        if 'cupy_total_mb' in initial_mem:
            print(f"CuPy Pool Used: {initial_mem['cupy_used_mb']:.2f} MB")
            print(f"CuPy Pool Free: {initial_mem['cupy_free_mb']:.2f} MB")

    # Set default batch sizes if not provided
    if test_batch_sizes is None:
        test_batch_sizes = [500, 1000, 2000, 5000, 10000, 20000, 50000, 100000, 200000, 500000]
    
    optimal_batch_size = 500  # Default
    if is_cupy:
        print(f"\n=== Batch Size Optimization ===")
        
        batch_results = []
        oom_occurred = False
        baseline_time_per_sample = None
        
        for batch_size in test_batch_sizes:
            if batch_size > test_n_samples:
                continue
            
            print(f"\n--- Testing batch_size={batch_size} ---")
            
            mem_before = get_gpu_memory_info()
            
            try:
                start_time = time.time()
                prob = bmdp.η_n(
                    π_target, π_0, u, 
                    n_samples=test_n_samples, 
                    seed=42, 
                    batch_size=batch_size, 
                    show_progress=True
                )
                elapsed_time = time.time() - start_time
                
                if is_cupy:
                    np.cuda.Stream.null.synchronize()
                
                mem_after = get_gpu_memory_info()
                
                mem_delta_mb = 0
                if 'cupy_used_mb' in mem_before and 'cupy_used_mb' in mem_after:
                    mem_delta_mb = mem_after['cupy_used_mb'] - mem_before['cupy_used_mb']
                
                time_per_sample = elapsed_time / test_n_samples * 1000  # ms
                samples_per_second = test_n_samples / elapsed_time
                
                speedup_vs_baseline = None
                if baseline_time_per_sample is not None:
                    speedup_vs_baseline = baseline_time_per_sample / time_per_sample
                else:
                    baseline_time_per_sample = time_per_sample
                
                batch_results.append({
                    'batch_size': batch_size,
                    'time': elapsed_time,
                    'time_per_sample': time_per_sample,
                    'samples_per_second': samples_per_second,
                    'memory_delta_mb': mem_delta_mb,
                    'memory_used_mb': mem_after.get('cupy_used_mb', 0),
                    'memory_free_mb': mem_after.get('cupy_free_mb', 0),
                    'success': True
                })
                
                print(f"  ⚡ Performance Metrics:")
                print(f"     Total time: {elapsed_time:.4f}s")
                print(f"     Time per sample: {time_per_sample:.4f} ms/sample")
                print(f"     Throughput: {samples_per_second:.1f} samples/sec")
                if speedup_vs_baseline is not None:
                    print(f"     Speedup vs baseline: {speedup_vs_baseline:.2f}x")
                else:
                    print(f"     Speedup vs baseline: 1.00x (baseline)")
                
                if mem_delta_mb > 0:
                    print(f"  💾 Memory delta: {mem_delta_mb:.2f} MB")
                if 'cupy_free_mb' in mem_after:
                    print(f"  💾 CuPy free memory: {mem_after['cupy_free_mb']:.2f} MB")
                
            except Exception as e:
                error_msg = str(e)
                oom_occurred = True
                batch_results.append({
                    'batch_size': batch_size,
                    'success': False,
                    'error': error_msg
                })
                print(f"  ✗ Failed: {error_msg}")
                
                if "IllegalAddress" in error_msg or "ILLEGAL_ADDRESS" in error_msg:
                    print(f"  ⚠ GPU memory access error - possible GPU state corruption")
                    print(f"  💡 Try restarting the Jupyter kernel")
                elif "out of memory" in error_msg.lower() or "OOM" in error_msg:
                    print(f"  ⚠ Out of memory error - this batch size is too large")
                else:
                    print(f"  ⚠ Unexpected error - may indicate GPU state issues")
                
                print(f"  Stopping batch size tests")
                break
        
        # Analyze results to find optimal batch size
        if batch_results:
            successful_results = [r for r in batch_results if r.get('success', False)]
            
            if successful_results:
                print(f"\n=== Batch Size Analysis ===")
                print(f"{'Batch Size':<15s} {'Time/sample (ms)':<20s} {'Memory (MB)':<20s} {'Speedup':<15s}")
                print(f"{'-'*70}")
                
                fastest = min(successful_results, key=lambda x: x['time_per_sample'])
                baseline = successful_results[0]
                
                for result in successful_results:
                    speedup = baseline['time_per_sample'] / result['time_per_sample']
                    mem_str = f"{result.get('memory_delta_mb', 0):.1f} Δ"
                    speedup_str = f"{speedup:.2f}x" if result != baseline else "1.00x"
                    print(f"{result['batch_size']:<15d} {result['time_per_sample']:<20.4f} {mem_str:<20s} {speedup_str:<15s}")
                
                optimal_candidates = [
                    r for r in successful_results 
                    if r['time_per_sample'] <= fastest['time_per_sample'] * 1.1
                ]
                
                if optimal_candidates:
                    optimal = max(optimal_candidates, key=lambda x: x['batch_size'])
                    optimal_batch_size = optimal['batch_size']
                    
                    print(f"\n=== Recommendations ===")
                    print(f"✓ Optimal batch size: {optimal_batch_size}")
                    print(f"  - Time per sample: {optimal['time_per_sample']:.4f} ms")
                    print(f"  - Speedup vs baseline: {baseline['time_per_sample'] / optimal['time_per_sample']:.2f}x")
                    if optimal.get('memory_delta_mb', 0) > 0:
                        print(f"  - Memory usage: {optimal['memory_delta_mb']:.2f} MB")
                    
                    if not oom_occurred and optimal_batch_size == successful_results[-1]['batch_size']:
                        print(f"  ⚠ Consider testing even larger batch sizes (memory allows)")
                    elif oom_occurred:
                        print(f"  ⚠ Larger batch sizes cause OOM - this is near the limit")
                else:
                    optimal_batch_size = fastest['batch_size']
                    print(f"\n=== Recommendations ===")
                    print(f"✓ Fastest batch size: {optimal_batch_size}")
            else:
                print(f"\n⚠ No successful batch size tests - using default batch_size=500")
        else:
            print(f"\n⚠ No batch size results - using default batch_size=500")
    else:
        print(f"\n⚠ CuPy not available - using default batch_size=500")
    
    print(f"\n=== Final Recommendation ===")
    print(f"✓ Optimal batch size: {optimal_batch_size}")
    
    return optimal_batch_size


# Run batch size optimization test
# This uses the adequate n_samples determined from the convergence test above
print("\n" + "="*70)
print("Running batch size optimization test for Localization...")
print("="*70)
optimal_batch_size = test_optimal_batch_size(quantization_level=2)


In [ ]:
def test_f_h_eta_consistency():
    """
    Test F, H, and η_n mathematical consistency for localization.
    """
    obstacles, area = load_obstacles_config(environment='toy2')
    motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
    sensor = LIDAR(fov=360, r_max=10.0, B=8)
    grid_map = LidarGridMapVec(
        x_min=area[0], x_max=area[1],
        y_min=area[2], y_max=area[3],
        quantization_level=3,
    )
    bmdp = BeliefMDP_n_Localization(
        n=3,
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=obstacles,
        _map=grid_map,
    )
    bmdp.set_known_map(obstacles)

    m_n = bmdp.SQ.m_n

    # Create uniform belief (1D for localization)
    π_0 = random.dirichlet(np.ones(m_n)).flatten()
    π_0 = π_0 / π_0.sum()

    u = np.array([0.5, 0.0])
    B = sensor.B
    r_max = sensor.r_max

    print(f"\n=== Testing F/H/η_n consistency (Localization) ===")

    # Test F consistency: posterior should be valid
    y_test = random.uniform(0.1, r_max, B)
    π_post = bmdp.F_batch(π_0, u, y_test[np.newaxis, :])[0]

    assert np.all(π_post >= 0), "Posterior must be non-negative"
    assert np.abs(π_post.sum() - 1.0) < 1e-10, "Posterior must be normalized"
    print("✓ F produces valid normalized beliefs")

    # Test H normalization
    H_vals = bmdp.H_vectorized_batch(y_test[np.newaxis, :], π_0, u)
    print(f"✓ H computation successful: H(y|π,u) = {float(H_vals[0]):.6e}")

    print("✓ F, H, η_n are mathematically consistent")

test_f_h_eta_consistency()
